# ORATS 公开近收盘样本：SOXX 25Δ 来源预检

仅解包官方公开 2024-01-03 ZIP，核 SOXX 原始行、双边报价、IV 与 Delta；不计算历史因子、读取 SOXX 收益或运行账户。

In [1]:
import os,sys,json,hashlib,zipfile,io,csv,datetime,math,collections,tarfile,tempfile,subprocess,shlex
from pathlib import Path
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_orats_public_soxx_sample_20261023';snap=root/'data/snapshots/stage1_orats_public_soxx_sample_20261023';plan=json.loads((card/'plan.json').read_text());head=json.loads((card/'head_receipt.json').read_text());state=json.loads((card/'run_state.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and head['HTTP_status']==200 and head['content_type']=='application/zip' and head['content_length']<=plan['budget']['max_compressed_bytes'] and head['final_url']==plan['exact_sample_url']
assert state['HEAD_attempts']==1 and state['GET_attempts']==1 and state['plan_sha256']==sha(card/'plan.json')
raw=snap/'ORATS_SMV_Strikes_20240103.zip';assert raw.is_file() and raw.stat().st_size==head['content_length'] and zipfile.is_zipfile(raw)
raw_digest=sha(raw);total=0;soxx=[];members=[];header=None
with zipfile.ZipFile(raw) as z:
 members=[{'name':i.filename,'uncompressed_bytes':i.file_size,'compressed_bytes':i.compress_size} for i in z.infolist()]
 assert len(members)==1 and members[0]['name']=='ORATS_SMV_Strikes_20240103.csv'
 with io.TextIOWrapper(z.open(members[0]['name']),encoding='utf-8-sig',newline='') as f:
  reader=csv.DictReader(f);header=reader.fieldnames
  required={'ticker','trade_date','expirDate','strike','cOpra','pOpra','cBidPx','cAskPx','pBidPx','pAskPx','cMidIv','pMidIv','delta','stkPx'}
  assert required.issubset(header)
  for row in reader:
   total+=1
   if row['ticker']=='SOXX':soxx.append(row)
assert total>500000 and len(soxx)>100 and all(r['trade_date']=='1/3/2024' for r in soxx)
trade=datetime.date(2024,1,3)
def num(row,key):
 try:return float(row[key])
 except (ValueError,TypeError):return math.nan
expiries={};valid_call=valid_put=0
for row in soxx:
 exp=datetime.datetime.strptime(row['expirDate'],'%m/%d/%Y').date();dte=(exp-trade).days;call_delta=num(row,'delta');put_delta=call_delta-1
 cb,ca,pb,pa,civ,piv=[num(row,k) for k in ('cBidPx','cAskPx','pBidPx','pAskPx','cMidIv','pMidIv')]
 cgood=all(math.isfinite(v) for v in (cb,ca,civ,call_delta)) and cb>0 and ca>=cb and civ>0 and 0<=call_delta<=1
 pgood=all(math.isfinite(v) for v in (pb,pa,piv,put_delta)) and pb>0 and pa>=pb and piv>0 and -1<=put_delta<=0
 valid_call+=cgood;valid_put+=pgood
 item={'strike':num(row,'strike'),'call_delta':call_delta,'put_delta':put_delta,'call_bid':cb,'call_ask':ca,'call_mid_iv':civ,'put_bid':pb,'put_ask':pa,'put_mid_iv':piv,'call_quote_valid':bool(cgood),'put_quote_valid':bool(pgood),'call_OPRA':row['cOpra'],'put_OPRA':row['pOpra']}
 expiries.setdefault((exp.isoformat(),dte),[]).append(item)
def nearest_and_bracket(rows,side,target):
 good=sorted([x for x in rows if x[side+'_quote_valid']],key=lambda x:x[side+'_delta']);above=[x for x in good if x[side+'_delta']>=target];below=[x for x in good if x[side+'_delta']<=target]
 near=min(good,key=lambda x:abs(x[side+'_delta']-target)) if good else None
 def compact(x):return None if x is None else {k:x[k] for k in ('strike',side+'_delta',side+'_bid',side+'_ask',side+'_mid_iv',side+'_OPRA')}
 return {'valid_rows':len(good),'nearest':compact(near),'lower':compact(below[-1] if below else None),'upper':compact(above[0] if above else None)}
expiry_report=[]
for (date,dte),rows in sorted(expiries.items(),key=lambda x:x[0][1]):
 expiry_report.append({'expiry':date,'calendar_days':dte,'strike_rows':len(rows),'call25':nearest_and_bracket(rows,'call',.25),'put_minus25':nearest_and_bracket(rows,'put',-.25)})
bracketing=[e for e in expiry_report if e['calendar_days']<=30];later=[e for e in expiry_report if e['calendar_days']>=30]
nearest_tenors=[bracketing[-1]['calendar_days'] if bracketing else None,later[0]['calendar_days'] if later else None]
report={'decision':'ORATS_public_2024_SOXX_strike_chain_schema_verified_only','plan_sha256':sha(card/'plan.json'),'source_URL':plan['exact_sample_url'],'raw_ZIP_sha256':raw_digest,'raw_ZIP_bytes':raw.stat().st_size,'members':members,'CSV_columns':header,'CSV_rows_all_symbols':total,'SOXX_strike_rows':len(soxx),'SOXX_expirations':expiry_report,'SOXX_valid_call_quote_IV_Delta_rows':int(valid_call),'SOXX_valid_put_quote_IV_Delta_rows':int(valid_put),'30_day_bracketing_tenors':nearest_tenors,'ORATS_delta_interpretation':'CSV delta is call Delta; vendor says put Delta equals call Delta minus 1.','delta_definition_source':'https://orats.com/near-eod-data','sample_date':plan['sample_date'],'2011_SOXX_specific_coverage_qualified':False,'historical_first_delivery_qualified':False,'full_calendar_missingness_qualified':False,'25delta_full_history_feature_qualified':False,'HEAD_requests':1,'GET_requests':1,'purchases':0,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(report,indent=2)+'\n')
settings=_settings(root);files=[raw,card/'plan.json',card/'head_receipt.json',card/'source_audit.json']
with tempfile.TemporaryDirectory(prefix='orats-soxx-public-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for p in files:out.add(p,arcname=str(p.relative_to(root)))
 digest=sha(archive);response=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research');uploaded=response['artifact'];assert uploaded['sha256']==digest and uploaded['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;restored=Path(tmp)/'restored.tar.gz'
 with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=180)
 assert run.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()};assert set(names)=={str(p.relative_to(root)) for p in files}
  for p in files:assert hashlib.sha256(z.extractfile(names[str(p.relative_to(root))]).read()).hexdigest()==sha(p)
archive_receipt={'sha256':digest,'size_bytes':uploaded['size_bytes'],'nas_path':remote,'members':{str(p.relative_to(root)):sha(p) for p in files},'remote_and_scoped_restore_verified':True};(card/'nas_source_archive.json').write_text(json.dumps(archive_receipt,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'source_audit':'source_audit.json','source_archive':'nas_source_archive.json','SOXX_strike_rows':len(soxx),'SOXX_valid_call_rows':int(valid_call),'SOXX_valid_put_rows':int(valid_put),'2011_SOXX_specific_coverage_qualified':False,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Single 2024 near-EOD sample only; 2011 SOXX history and delivery clock not proven.','The paired-strike delta column is call delta; put delta is reconstructed per vendor definition.','No historical factor, model or fee-net account.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='finished_source_only';state['raw_sha256']=raw_digest;state['NAS_sha256']=digest;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:report[k] for k in ('decision','CSV_rows_all_symbols','SOXX_strike_rows','SOXX_valid_call_quote_IV_Delta_rows','SOXX_valid_put_quote_IV_Delta_rows','30_day_bracketing_tenors')})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'decision': 'ORATS_public_2024_SOXX_strike_chain_schema_verified_only', 'CSV_rows_all_symbols': 716822, 'SOXX_strike_rows': 1124, 'SOXX_valid_call_quote_IV_Delta_rows': 1050, 'SOXX_valid_put_quote_IV_Delta_rows': 957, '30_day_bracketing_tenors': [16, 44]}


# ORATS 官方公开样本：SOXX 25Δ 行权价链存在

[ORATS 官方历史数据页](https://orats.com/university/historical-data)公开的 2024-01-03 近收盘 ZIP，经冻结1次 HEAD、1次 GET 获取；压缩原件66,808,802字节、SHA-256 `e04a3731…`，716,822条全市场行权价记录中 **1,124条为 SOXX**。Yanxuan NAS 归档 SHA-256 `a91c6d1b…` 已远端哈希与限定恢复；原件可按收据恢复，不依赖未归档本地缓存。零购买、零 SOXX 未来收益读取、零拟合、零账户。

SOXX 样本中1,050条看涨和957条看跌行同时具备正值双边报价、正值mid IV与有效 Delta。[ORATS 字段说明](https://orats.com/near-eod-data)规定该单列 `delta` 是看涨 Delta，看跌 Delta 为其减1。距离30日最近的两个月期限分别是 **16日和44日**；两期限上，+0.25看涨和−0.25看跌均有有效报价与IV的上下行权价括号。具体行权价、bid/ask、IV、Delta与合约标识见 `source_audit.json`。这证明**单日 SOXX 链的特征可构造性**，未计算或检验收益预测。

**来源仍未通过历史门。** 当前样本只有2024-01-03，不能推断2011–2022 SOXX逐日覆盖、缺失与拆分后合约身份；CSV只有交易日期，没有每行首次报价/文件送达时间。ORATS产品说明称近收盘数据夜间交付，但未认证每个历史日期在下一交易日 Stage1 09:46 前已按相同数值可得。已更新[只询样邮件草稿](../soxx_option_vendor_sample_preflight_20261023/orats_inquiry_draft.md)以集中请求2011/2012原始 SOXX 样本和发布/修订时钟，尚未发送或购买。之后只有完整来源通过，才可冻结25Δ约30日偏斜的匹配信息门与完整Stage1 0bp Tiered账户。
